# 13_prep_receptor — 도킹 수용체(단백질) 준비

**한 줄 요약:** HSD17B13 단백질 구조(PDB **8G89**)를 내려받아, 물은 빼고 보조인자(NAD)는 남기고, 결합해 있던 저해제는 따로 분리해 도킹용 수용체 파일을 만든다.
**용어:** PDB=단백질 3D 구조 파일 / 보조인자(NAD)=효소가 일할 때 필요한 분자 / PDBQT=도킹용 단백질 파일.
**큰 흐름:** ① 준비 → ② 잔기 분류표·도구찾기 → ③ 다운로드 → ④ 원자 파싱 → ⑤ 저해제 선택 → ⑥ 파일 작성 → ⑦ PDBQT 변환

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + 경로 설정
파일 다운로드·명령 실행에 필요한 라이브러리를 가져오고 저장 경로를 정한다.

In [ ]:
import os
import sys
import shutil
import subprocess
import urllib.request
from collections import defaultdict

OUTDIR = "data/docking/receptor"
PDB_ID = "8G89"
os.makedirs(OUTDIR, exist_ok=True)
RAW = os.path.join(OUTDIR, f"{PDB_ID}.pdb")
REC_PDB = os.path.join(OUTDIR, "receptor.pdb")
REC_PDBQT = os.path.join(OUTDIR, "receptor.pdbqt")
REF_LIG = os.path.join(OUTDIR, "ref_ligand.pdb")

🔎 **코드 뜯어보기 (셀 1)**
- `import urllib.request` : 인터넷에서 **파일 다운로드**. `import subprocess`=외부 프로그램 실행. `import shutil`=프로그램 위치 찾기. `from collections import defaultdict`=값이 자동으로 리스트로 시작되는 딕셔너리.

### 셀 2 — 잔기 분류 목록 + 도구 찾기 함수
무엇을 남기고(단백질·NAD) 버릴지(물·이온) 정한 목록과, 설치된 프로그램을 찾는 함수를 만든다.

In [ ]:
COFACTORS = {"NAD", "NAI", "NAP", "NDP", "NAJ", "NAX"}
DROP_HET = {"HOH", "WAT", "DOD",
            "GOL", "EDO", "PEG", "PGE", "PG4", "1PE", "ACT", "FMT", "DMS", "MPD",
            "SO4", "PO4", "CL", "NA", "K", "MG", "CA", "ZN", "MN", "IOD", "BR",
            "TRS", "EPE", "IMD", "BME", "CIT", "MES"}
AA = {"ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
      "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
      "MSE", "SEC", "PYL"}


def which(name):
    return shutil.which(name) or shutil.which(name + ".exe")

🔎 **코드 뜯어보기 (셀 2)**
- `COFACTORS = {"NAD", ...}` : **집합**(`{ }`) — 남길 보조인자 이름들. `DROP_HET`=버릴 것, `AA`=아미노산(단백질).
- `def which(name): return shutil.which(name) or shutil.which(name + ".exe")` : 프로그램 경로를 찾음(`or`=앞이 없으면 뒤).

### 셀 3 — 8G89 구조 다운로드
RCSB(단백질 구조 사이트)에서 8G89.pdb 파일을 내려받는다(이미 있으면 건너뜀).

In [ ]:
if not os.path.exists(RAW):
    url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
    print(f"다운로드: {url}")
    urllib.request.urlretrieve(url, RAW)
print(f"수용체 원본: {RAW}")

🔎 **코드 뜯어보기 (셀 3)**
- `if not os.path.exists(RAW):` : 파일이 없을 때만 다운로드. `urllib.request.urlretrieve(url, RAW)`=url의 파일을 RAW 경로로 저장.

### 셀 4 — PDB 파일의 원자들을 분류
한 줄씩 읽어 단백질/보조인자(NAD)/저해제 후보로 나눈다.

In [ ]:
with open(RAW) as f:
    lines = f.readlines()

het_atoms = defaultdict(list)   # (resname, chain, resseq) -> lines
het_count = defaultdict(int)    # resname -> atom 수
protein_lines, cofactor_lines = [], []
for ln in lines:
    rec = ln[:6].strip()
    if rec not in ("ATOM", "HETATM"):
        continue
    resn = ln[17:20].strip()
    chain = ln[21:22]
    resseq = ln[22:26].strip()
    if rec == "ATOM" or resn in AA:
        protein_lines.append(ln)
    elif resn in COFACTORS:
        cofactor_lines.append(ln)
    elif resn in DROP_HET:
        continue
    else:
        het_atoms[(resn, chain, resseq)].append(ln)
        het_count[resn] += 1

print("\n발견된 비표준 헤테로 잔기(물·이온·첨가물 제외):")
for (resn, ch, seq), lns in sorted(het_atoms.items(), key=lambda kv: -len(kv[1])):
    print(f"  {resn} (chain {ch} #{seq}): {len(lns)} atoms")
print(f"보조인자(유지): {sorted(set(l[17:20].strip() for l in cofactor_lines)) or '없음'}")

if not het_atoms:
    print("\n[경고] 저해제 후보를 못 찾음. PDB의 HET 목록을 확인해 수동 지정 필요.")
    sys.exit(1)

🔎 **코드 뜯어보기 (셀 4)**
- `with open(RAW) as f: lines = f.readlines()` : 파일을 열어 **모든 줄**을 리스트로 읽기.
- `ln[17:20]` : 문자열의 **일부 잘라내기**(PDB는 위치가 정해져 있어 17~19번째 글자가 잔기 이름). `.strip()`=공백 제거.
- `het_atoms[(resn, chain, resseq)].append(ln)` : defaultdict라 키가 없어도 자동으로 리스트를 만들어 추가.
- `sorted(..., key=lambda kv: -len(kv[1]))` : 원자 수 많은 순으로 정렬(`-`로 내림차순).

### 셀 5 — 공결정 저해제 고르기
남은 헤테로 잔기 중 **가장 큰 것**을 결합해 있던 저해제로 본다(도킹 기준·양성대조).

In [ ]:
ref_key = max(het_atoms, key=lambda k: len(het_atoms[k]))
print(f"\n기준 저해제로 선택: {ref_key[0]} (chain {ref_key[1]} #{ref_key[2]}, "
      f"{len(het_atoms[ref_key])} atoms)  ← 틀리면 스크립트 상단에서 조정")

🔎 **코드 뜯어보기 (셀 5)**
- `max(het_atoms, key=lambda k: len(het_atoms[k]))` : 딕셔너리에서 **값(원자 수)이 가장 많은 키**를 고르기 → 가장 큰 저해제. `sys.exit(1)`=문제 있으면 중단.

### 셀 6 — 수용체·기준리간드 파일 작성
단백질+NAD는 수용체로, 저해제는 기준 리간드로 각각 저장한다.

In [ ]:
with open(REC_PDB, "w") as f:
    f.writelines(protein_lines + cofactor_lines)
    f.write("END\n")
with open(REF_LIG, "w") as f:
    f.writelines(het_atoms[ref_key])
    f.write("END\n")
print(f"수용체(단백질+NAD): {REC_PDB}")
print(f"기준 리간드: {REF_LIG}")

🔎 **코드 뜯어보기 (셀 6)**
- `with open(REC_PDB, "w") as f: f.writelines(protein_lines + cofactor_lines)` : 쓰기(w) 모드로 열어 단백질+NAD 줄들을 파일에 씀. `+`=두 리스트 이어붙이기.

### 셀 7 — OpenBabel로 수용체 PDBQT 변환
수소·전하를 붙여 도킹용 형식(PDBQT)으로 바꾼다(OpenBabel 필요).

In [ ]:
obabel = which("obabel")
if not obabel:
    print("\n[다음 단계 필요] OpenBabel 미설치. 설치 후 아래 실행:")
    print("  conda install -c conda-forge openbabel")
    print(f'  obabel "{REC_PDB}" -O "{REC_PDBQT}" -xr -p 7.4')
    sys.exit(0)
cmd = [obabel, REC_PDB, "-O", REC_PDBQT, "-xr", "-p", "7.4"]
print("\n실행:", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stderr.strip()[-300:] if r.stderr else "")
if os.path.exists(REC_PDBQT) and os.path.getsize(REC_PDBQT) > 0:
    print(f"수용체 PDBQT 완료: {REC_PDBQT}")
    print("\n→ 다음: python scripts/14_run_docking.py")
else:
    print("[실패] 수용체 PDBQT 생성 실패. OpenBabel 출력 확인.")

🔎 **코드 뜯어보기 (셀 7)**
- `obabel = which("obabel")` : OpenBabel이 깔려 있나 확인. 없으면 안내 후 `sys.exit(0)`으로 정상 종료.
- `subprocess.run(cmd, capture_output=True, text=True)` : **외부 프로그램(OpenBabel) 실행**. cmd는 [프로그램, 인자들] 리스트. `-xr`=강체 수용체, `-p 7.4`=pH 7.4로 수소 처리.
- `os.path.getsize(REC_PDBQT) > 0` : 결과 파일이 비어있지 않은지 확인.